# CH13 — the SYSMON: a potentiometer, a die temperature, and six supply rails

The AUP-ZU3 wires a 10K potentiometer to **VP (R13)** and **VN (T12)**, the
dedicated analog input pins of the UltraScale+ **SYSMONE4**. On 7-series parts
this block is called XADC; here it is reached through the **System Management
Wizard**.

This notebook reads the pot, shows it as a moving bar, mirrors it onto the
board's eight white LEDs through a hardware thermometer decoder, and reads the
built-in temperature and supply sensors alongside it.

In [ ]:
import sys, time, pathlib

# This notebook runs from two places, so it searches rather than assumes:
#   ~/jupyter_notebooks/handbook/CH13/project_xadc_sysmon/notebooks
#       -> drivers two levels up, in the chapter's shared CH13/sw
#   ~/ch13_xadc/notebooks   (the bare test harness)
#       -> drivers one level up
_here = pathlib.Path.cwd()

def _find(candidates, what):
    for rel in candidates:
        cand = (_here / rel).resolve()
        if cand.exists():
            return cand
    raise FileNotFoundError(
        "cannot find %s -- looked for %s relative to %s"
        % (what, candidates, _here))

SW  = _find(["../../sw", "../sw", "sw"], "the sw/ directory")
BIT = _find(["../out/xadc_sysmon.bit", "../../out/xadc_sysmon.bit",
             "out/xadc_sysmon.bit"], "xadc_sysmon.bit")
sys.path.insert(0, str(SW))
print("drivers  :", SW)
print("bitstream:", BIT)

from pynq import Overlay, MMIO
import sysmon
from sysmon import Sysmon, Activity, PotBar

ov = Overlay(str(BIT))

# The AXI window is 8 KB. Mapping only 4 KB would leave the DRP registers,
# which start at 0x1400, outside the mapping entirely.
sm  = Sysmon(MMIO(sysmon.SYSMON_BASE,    sysmon.AXI_WINDOW_SIZE))
act = Activity(MMIO(sysmon.ACTIVITY_BASE, 0x1000))
bar = PotBar(MMIO(sysmon.POT_GPIO_BASE,   0x1000))

print("SYSMON converting:", sm.is_converting())

## Everything the SYSMON measures, once

`read_all()` takes one pass so the numbers belong to the same moment. The
min/max temperatures are latched by the macro itself since power-up — software
never has to poll to catch a transient.

In [ ]:
r = sm.read_all()

print("die temperature   %7.2f C" % r["temperature"])
lo, hi = sm.temperature_extremes_celsius()
print("min / max         %7.2f / %.2f C   (latched by the macro)" % (lo, hi))

print("\nsupply rails")
for name, v in r["supplies"].items():
    print("  %-11s %8.4f V" % (name, v))

e = r["external"]
print("\npot on VP/VN      raw 0x%04X   code %d/1023   %.4f V   %.1f%% of full scale"
      % (e["raw"], e["code"], e["volts"], 100.0 * e["fraction"]))

## The pot, liveTurn the potentiometer. The bar follows it, and so do the eight white LEDs —those go through `hdl/pot_bar.sv`, a thermometer decoder in the PL.The value reaches that decoder from software rather than straight from theSYSMON, and that is forced rather than chosen: with `INTERFACE_SELECTION =Enable_AXI` the wizard owns the SYSMONE4's **single** DRP port, so fabric logiccannot read conversions out of the macro. The decode stays in hardware; onlythe value's journey goes through the PS.**The bar is normalised to the pot's measured travel.** Swept end to end this

board's wiper spans 0 to `0xDAC1` — 0 to 0.8545 V, 85.5% of the channel's

range — so a bar scaled to the ADC's full scale would only ever reach seven of

its eight LEDs. `normalize_pot()` rescales the display so a full turn fills it.

The volts shown stay the true measurement.



**VP/VN is unipolar with a fixed 1.0 V full scale.** It is not adjustable, so apot whose divider exceeded 1.0 V would clip rather than scale. This board'sdoes not — measured, the wiper spans well inside the range.Press **Stop** to end it. The sampling runs on a background thread, because
a plain `while` loop in a cell blocks the kernel and the button click would
never be delivered — which is exactly how the first version of this cell got
a Stop button that did nothing.

In [ ]:
import threading
import ipywidgets as widgets
from IPython.display import display

# A plain `while` loop in a notebook cell BLOCKS THE KERNEL. While it spins,
# the kernel cannot process the messages the browser sends, so a button click
# never arrives, its handler never runs, and the loop never sees the flag it
# is waiting for. The first version of this cell did exactly that: the Stop
# button did nothing and only Kernel > Interrupt got you out.
#
# So the sampling runs on a background thread and this cell returns straight
# away, which leaves the kernel free to handle the click.

# Stop a sampler left over from a previous run of this cell. Two threads
# writing the same widgets and the same GPIO is not worth debugging.
try:
    _stop.set()
    _sampler.join(timeout=2.0)
except NameError:
    pass

meter    = widgets.FloatProgress(value=0.0, min=0.0, max=1.0,
                                 description="VP/VN", bar_style="info",
                                 layout=widgets.Layout(width="70%"))
readout  = widgets.HTML()
sensors  = widgets.HTML()
stop_btn = widgets.Button(description="Stop", icon="stop",
                          button_style="danger")

_stop = threading.Event()
stop_btn.on_click(lambda _b: _stop.set())


def _sample():
    while not _stop.is_set():
        raw = sm.vp_vn_raw()
        meter.value = sysmon.raw_to_external_volts(raw)
        readout.value = ("<b>%.4f V</b> &nbsp;&nbsp; raw <code>0x%04X</code>"
                         " &nbsp;&nbsp; code %d/1023 &nbsp;&nbsp;"
                         " %.1f%% of full scale"
                         % (meter.value, raw, sysmon.adc_code(raw),
                            100.0 * raw / 65536.0))
        sensors.value = ("die %.2f &deg;C &nbsp;&nbsp; VCCINT %.4f V"
                         " &nbsp;&nbsp; VCCAUX %.4f V"
                         % (sm.temperature_celsius(),
                            sm.supply_volts("vccint"),
                            sm.supply_volts("vccaux")))
        bar.set_reading(raw)   # mirror onto the LEDs, rescaled to the travel
        time.sleep(0.05)
    sensors.value += " &nbsp;&nbsp; <b>stopped</b>"


_sampler = threading.Thread(target=_sample, daemon=True)
display(widgets.VBox([meter, readout, sensors, stop_btn]))
_sampler.start()


## Is the ADC actually running?

This is here because of how the chapter went wrong.

Every SYSMON register read `0x0000` — temperature included, which is
physically impossible, since raw 0 decodes to −280 °C. Writes to the config
registers did not stick. Linux's own `xilinx-ams` driver reported
`in_temp20_raw = 0` for the PL while the PS sensors read correctly. It looked
conclusively like a SYSMON that was present, placed, and dead.

It was not. **The DRP window is at `0x1400`, not `0x200`.** `0x200` is the
*7-series XADC Wizard's* base; this IP's AXI window is 13 bits wide and puts
the DRP somewhere else entirely. Reads at `0x200` landed on nothing.

What settled it is below. The wizard brings the macro's own `eoc_out`,
`eos_out`, `busy_out` and `channel_out` pins into the fabric, where
`hdl/sysmon_activity.sv` counts them. That path shares nothing with the DRP or
with the PS AMS block, so it could still answer *is the converter running* when
every register read zero — and it showed several thousand conversions a second.

The counting has to happen in fabric: `eoc_out` is a single-cycle pulse at
100 MHz, and software polling a GPIO would miss essentially all of them and
report a dead converter either way.

**When an entire register window reads zero, question the address before
concluding the hardware is dead.**

In [ ]:
moving, before, after = act.is_converting(settle=1.0)
delta = (after["eoc_count"] - before["eoc_count"]) & 0xFFFFFFFF

print("conversions in 1.0 s : %d" % delta)
print("sequences completed  : %d" % ((after["eos_count"] - before["eos_count"]) & 0xFFFF))
print("channel now          : %d" % after["channel"])
print("verdict              : %s" % ("the ADC is running" if moving else "the ADC is STOPPED"))